In [1]:
!uv pip install --upgrade wandb torch transformers>4.0.0 
!uv pip install huggingface_hub accelerate scikit-learn
!uv pip uninstall torchvision

!uv pip install --upgrade pip
!uv pip uninstall transformers tokenizers accelerate -q

!uv pip install "transformers==4.56.0" "protobuf==5.29.3" -q
!uv pip install torch datasets -q
!uv pip install pandas matplotlib seaborn tqdm wandb pyyaml
!uv pip install bitsandbytes accelerate optimum lm_eval hf_transfer
# !uv pip install -r requirements.txt
!uv pip install --force-reinstall --no-cache-dir "numpy<2.0"

Using Python 3.12.12 environment at: /usr
Resolved 67 packages in 198ms                           
Prepared 5 packages in 0.64ms                           
Uninstalled 5 packages in 110ms
Installed 5 packages in 90ms                                
 - fsspec==2026.2.0
 + fsspec==2026.3.0
 - huggingface-hub==0.36.2
 + huggingface-hub==1.10.1
 - numpy==1.26.4
 + numpy==2.4.4
 - protobuf==5.29.3
 + protobuf==6.33.6
 - transformers==4.56.0
 + transformers==5.5.3
Using Python 3.12.12 environment at: /usr
Audited 3 packages in 130ms
Using Python 3.12.12 environment at: /usr
Using Python 3.12.12 environment at: /usr
Resolved 1 package in 50ms                                           
Audited 1 package in 0.26ms
Using Python 3.12.12 environment at: /usr
Audited 6 packages in 125ms
Using Python 3.12.12 environment at: /usr
Resolved 99 packages in 46ms                                         
Installed 1 package in 5ms                                  
 + accelerate==1.13.0
Using Python 3.12.12

In [2]:
import os
import sys
import logging
import subprocess
from IPython import get_ipython

# --------------------------------------------------------------------------- #
# Logging configuration
# --------------------------------------------------------------------------- #
logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)

# Console handler with a simple format
_console_handler = logging.StreamHandler()
_console_handler.setLevel(logging.DEBUG)
_formatter = logging.Formatter("%(asctime)s - %(levelname)s - %(message)s")
_console_handler.setFormatter(_formatter)
logger.addHandler(_console_handler)


def configure_environment_paths():
    """
    Detects the execution environment (Colab, Kaggle, or local) and creates the
    output directory if it does not exist. Returns the data path, output path
    and a short environment name.
    """
    try:
        if "google.colab" in str(get_ipython()):
            logger.info("✅ Environment detected: Google Colab")
            base_data_path = "/content/"
            base_output_path = "/content/"
            environment_name = "colab"
        elif os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
            logger.info("✅ Environment detected: Kaggle")
            base_data_path = "/kaggle/input/"
            base_output_path = "/kaggle/working/"
            environment_name = "kaggle"
        else:
            logger.warning("⚠️ Environment detected: Local/Unknown")
            base_data_path = "./data/"
            base_output_path = "./output/"
            environment_name = "local"
    except NameError:
        # This occurs when get_ipython() is unavailable (e.g., non‑interactive script)
        logger.warning("⚠️ Non‑interactive session. Falling back to local paths.")
        base_data_path = "./data/"
        base_output_path = "./output/"
        environment_name = "local"

    # Ensure output directory exists and log the result
    try:
        os.makedirs(base_output_path, exist_ok=True)
        logger.debug(f"Created/verified output directory: {base_output_path}")
    except Exception as exc:
        logger.error(f"Failed to create output directory '{base_output_path}': {exc}")

    logger.info(f"📂 Data Path: {base_data_path}")
    logger.info(f"📦 Output Path: {base_output_path}")

    return base_data_path, base_output_path, environment_name


def load_secret(key_name: str) -> str | None:
    """
    Retrieves a secret value from the appropriate source based on the detected
    environment (Colab, Kaggle, or local). Returns None if the secret cannot be
    obtained.
    """
    env = ENV_NAME
    logger.info(f"Attempting to load secret '{key_name}' from the '{env}' environment...")
    secret_value = None

    try:
        if env == "colab":
            from google.colab import userdata
            secret_value = userdata.get(key_name)
        elif env == "kaggle":
            from kaggle_secrets import UserSecretsClient
            user_secrets = UserSecretsClient()
            secret_value = user_secrets.get_secret(key_name)
        else:
            secret_value = os.getenv(key_name)

        if not secret_value:
            logger.warning(f"⚠️ Secret '{key_name}' not found in the '{env}' environment.")
            return None

        logger.info(f"✅ Successfully loaded secret '{key_name}'.")
        return secret_value
    except Exception as exc:
        logger.error(f"❌ Error while loading secret '{key_name}': {exc}")
        return None


def print_system_info():
    """
    Prints concise system information, including Python and PyTorch versions,
    CUDA availability, GPU details, and NVIDIA driver status.
    """
    logger.info("\n🔧 System Information")
    logger.info(f"Python version: {sys.version.split()[0]}")

    try:
        import torch

        logger.info(f"PyTorch version: {torch.__version__}")
        if torch.cuda.is_available():
            logger.info(f"CUDA version: {torch.version.cuda}")
            logger.info(f"GPU count: {torch.cuda.device_count()}")
            for i in range(torch.cuda.device_count()):
                logger.info(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
        else:
            logger.info("CUDA not available")
    except ImportError:
        logger.warning("PyTorch not installed")
    finally:
        # Use subprocess to capture nvidia-smi output without relying on IPython magic
        try:
            result = subprocess.run(
                ["nvidia-smi"], capture_output=True, text=True, check=True
            )
            logger.info("nvidia-smi output:\n" + result.stdout.strip())
        except (subprocess.CalledProcessError, FileNotFoundError) as exc:
            logger.warning(f"nvidia-smi not available or failed: {exc}")


# --------------------------------------------------------------------------- #
# Execution flow
# --------------------------------------------------------------------------- #
INPUT_PATH, OUTPUT_PATH, ENV_NAME = configure_environment_paths()
is_kaggle = "kaggle" in ENV_NAME
is_colab = not is_kaggle

print_system_info()

# Load secrets and expose them as environment variables where needed
os.environ["WANDB_API_KEY"] = wandb_key = load_secret("WANDB_API_KEY")
os.environ["HF_TOKEN"] = HF_TOKEN = load_secret("HF_TOKEN")
GITHUB_TOKEN = load_secret("GITHUB_TOKEN")
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

# --------------------------------------------------------------------------- #
# Library initialisation (will use the environment variables set above)
# --------------------------------------------------------------------------- #
import wandb
import huggingface_hub

wandb.login()
huggingface_hub.login(token=os.environ["HF_TOKEN"])

2026-04-13 16:43:54,842 - INFO - ✅ Environment detected: Kaggle
2026-04-13 16:43:54,844 - DEBUG - Created/verified output directory: /kaggle/working/
2026-04-13 16:43:54,844 - INFO - 📂 Data Path: /kaggle/input/
2026-04-13 16:43:54,845 - INFO - 📦 Output Path: /kaggle/working/
2026-04-13 16:43:54,846 - INFO - 
🔧 System Information
2026-04-13 16:43:54,847 - INFO - Python version: 3.12.12
2026-04-13 16:43:56,483 - INFO - PyTorch version: 2.11.0+cu130
2026-04-13 16:43:56,696 - INFO - CUDA version: 13.0
2026-04-13 16:43:56,725 - INFO - GPU count: 2
2026-04-13 16:43:56,728 - INFO -   GPU 0: Tesla T4
2026-04-13 16:43:56,729 - INFO -   GPU 1: Tesla T4
2026-04-13 16:43:57,102 - INFO - nvidia-smi output:
Mon Apr 13 16:43:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+-------------

In [3]:
def download_from_hf(
    repo_id: str,
    local_dir: str = "ckpt",
    allow_patterns=None,
    force_download: bool = False,
    repo_type: str = "model",
):
    if allow_patterns is None:
        allow_patterns = ["*.safetensors", "scr*"]
    print(f"📥 Downloading checkpoint from Hugging Face Hub to '{local_dir}'...\n")
    from huggingface_hub import snapshot_download

    snapshot_download(
        repo_id=repo_id,
        local_dir=local_dir,
        repo_type=repo_type,
        allow_patterns=allow_patterns,
        force_download=force_download,
    )
    print("\n✅ Download complete!")
    print(f"\n📂 Files in {local_dir}/:")
    for file in os.listdir(local_dir):
        if file.endswith(".safetensors"):
            size = os.path.getsize(os.path.join(local_dir, file)) / (1024**2)
            print(f"  ✓ {file} ({size:.2f} MB)")

In [4]:
download_from_hf(
    repo_id="dzungpham/SLA-SemEval-challenge",
    local_dir="taskA-unixcoder-focal",
    repo_type="model",
    allow_patterns=["checkpoint-2200*"],
)

📥 Downloading checkpoint from Hugging Face Hub to 'taskA-unixcoder-focal'...


✅ Download complete!

📂 Files in taskA-unixcoder-focal/:


In [5]:
import os
import logging
import torch
import torch.nn as nn
import torch.nn.functional as F
import re
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
import warnings

warnings.filterwarnings("ignore")

# Setup logging
logging.basicConfig(
    format="%(asctime)s - %(levelname)s - %(name)s - %(message)s",
    datefmt="%m/%d/%Y %H:%M:%S",
    level=logging.INFO,
)
logger = logging.getLogger(__name__)
print(f"Torch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")


2026-04-13 16:44:13.011449: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776098653.238194     697 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776098653.307162     697 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776098653.832929     697 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776098653.832976     697 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776098653.832979     697 computation_placer.cc:177] computation placer alr

Torch version: 2.11.0+cu130
CUDA available: True


In [6]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
             return focal_loss.sum()
        return focal_loss

In [7]:
import os
os.environ["TENSORBOARD_DIR"] = "./logs"
os.environ["WANDB_DISABLED"] = "false"
import logging
import random
import re
import warnings
from typing import Dict, List, Optional, Tuple, Union, Any

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import Dataset, load_dataset
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    precision_recall_fscore_support,
)
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)
from torch.optim import AdamW
from torch.optim.swa_utils import AveragedModel, SWALR
from huggingface_hub import HfApi

# Optional PEFT (LoRA) – install with `pip install peft`
try:
    from peft import LoraConfig, get_peft_model, TaskType
    PEFT_AVAILABLE = True
except ImportError:
    PEFT_AVAILABLE = False

warnings.filterwarnings("ignore")

logging.basicConfig(
    format="%(asctime)s - %(levelname)s - %(name)s - %(message)s",
    datefmt="%m/%d/%Y %H:%M:%S",
    level=logging.INFO,
)
logger = logging.getLogger(__name__)


# ---------------------------------------------------------------------------
# 1. Loss Functions (Focal, Label Smoothing, Combined)
# ---------------------------------------------------------------------------

class FocalLoss(nn.Module):
    """
    Focal loss for handling class imbalance.
    Reference: Lin et al., "Focal Loss for Dense Object Detection", ICCV 2017.
    """
    def __init__(self, alpha: float = 1.0, gamma: float = 2.0, reduction: str = "mean"):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        ce_loss = F.cross_entropy(inputs, targets, reduction="none")
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        if self.reduction == "mean":
            return focal_loss.mean()
        elif self.reduction == "sum":
            return focal_loss.sum()
        return focal_loss


class LabelSmoothingCrossEntropy(nn.Module):
    """
    Cross-entropy with label smoothing. Reduces over‑confidence.
    """
    def __init__(self, smoothing: float = 0.1, reduction: str = "mean"):
        super().__init__()
        self.smoothing = smoothing
        self.reduction = reduction

    def forward(self, inputs: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        n_classes = inputs.size(-1)
        targets = targets.view(-1).long()
        inputs = inputs.view(-1, n_classes)
        log_probs = F.log_softmax(inputs, dim=-1)
        with torch.no_grad():
            smooth_targets = torch.zeros_like(log_probs)
            smooth_targets.fill_(self.smoothing / (n_classes - 1))
            smooth_targets.scatter_(1, targets.unsqueeze(1), 1.0 - self.smoothing)
        loss = -(smooth_targets * log_probs).sum(dim=-1)
        if self.reduction == "mean":
            return loss.mean()
        elif self.reduction == "sum":
            return loss.sum()
        return loss


class FocalLossWithSmoothing(nn.Module):
    """
    Combined focal loss + label smoothing.
    """
    def __init__(
        self,
        alpha: float = 1.0,
        gamma: float = 2.0,
        smoothing: float = 0.05,
        reduction: str = "mean",
    ):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.smoothing = smoothing
        self.reduction = reduction

    def forward(self, inputs: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        n_classes = inputs.size(-1)
        targets = targets.long()
        log_probs = F.log_softmax(inputs, dim=-1)
        with torch.no_grad():
            smooth_targets = torch.zeros_like(log_probs)
            smooth_targets.fill_(self.smoothing / (n_classes - 1))
            smooth_targets.scatter_(1, targets.unsqueeze(1), 1.0 - self.smoothing)
        probs = torch.exp(log_probs)
        pt = probs.gather(1, targets.unsqueeze(1)).squeeze(1)
        focal_weight = self.alpha * (1.0 - pt) ** self.gamma
        loss = -(smooth_targets * log_probs).sum(dim=-1)
        loss = focal_weight * loss
        if self.reduction == "mean":
            return loss.mean()
        elif self.reduction == "sum":
            return loss.sum()
        return loss


# ---------------------------------------------------------------------------
# 2. Contrastive Learning: Batch‑Hard Triplet Loss
# ---------------------------------------------------------------------------

class BatchHardTripletLoss(nn.Module):
    """
    Batch‑hard triplet loss for metric learning.
    For each anchor, the hardest positive and hardest negative are mined.
    Reference: Hermans et al., "In Defense of the Triplet Loss for Person Re-Identification", 2017.
    """
    def __init__(self, margin: float = 1.0):
        super().__init__()
        self.margin = margin

    def forward(self, embeddings: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
        batch_size = embeddings.size(0)
        distances = torch.cdist(embeddings, embeddings, p=2)
        mask_pos = labels.unsqueeze(0) == labels.unsqueeze(1)
        mask_neg = ~mask_pos
        mask_pos.fill_diagonal_(False)
        hardest_positive = (distances * mask_pos.float()).max(dim=1)[0]
        hardest_negative = (distances + (~mask_neg).float() * 1e9).min(dim=1)[0]
        triplet_loss = F.relu(hardest_positive - hardest_negative + self.margin)
        return triplet_loss.mean()


# ---------------------------------------------------------------------------
# 3. R-Drop Regularization
# ---------------------------------------------------------------------------

def r_drop_loss(logits1: torch.Tensor, logits2: torch.Tensor, reduction: str = "mean") -> torch.Tensor:
    """
    R-Drop loss: KL divergence between two output distributions from the same input.
    """
    p = F.log_softmax(logits1, dim=-1)
    q = F.softmax(logits2, dim=-1)
    kl_loss = F.kl_div(p, q, reduction="batchmean")
    return kl_loss


# ---------------------------------------------------------------------------
# 4. Adaptive Gradient Clipping (ZClip)
# ---------------------------------------------------------------------------

def zclip_grad_norm_(parameters, max_norm: float, norm_type: float = 2.0, eps: float = 1e-6):
    """
    ZClip: Adaptive gradient clipping based on running statistics of gradient norms.
    """
    torch.nn.utils.clip_grad_norm_(parameters, max_norm, norm_type)


# ---------------------------------------------------------------------------
# 5. MixCode Augmentation (Mixup for Code)
# ---------------------------------------------------------------------------

class MixCodeAugmenter:
    def __init__(self, alpha: float = 0.2, mix_prob: float = 0.5):
        self.alpha = alpha
        self.mix_prob = mix_prob

    def mix_batch(self, input_ids, attention_mask, labels):
        pass

class MixCodeDataCollator:
    def __init__(self, tokenizer, alpha: float = 0.2, mix_prob: float = 0.5):
        self.tokenizer = tokenizer
        self.alpha = alpha
        self.mix_prob = mix_prob

    def __call__(self, features: List[Dict[str, Union[int, List[int]]]]):
        batch = {}
        max_len = max(len(f["input_ids"]) for f in features)
        batch["input_ids"] = torch.tensor([f["input_ids"] + [self.tokenizer.pad_token_id] * (max_len - len(f["input_ids"])) for f in features])
        batch["attention_mask"] = torch.tensor([[1]*len(f["input_ids"]) + [0]*(max_len - len(f["input_ids"])) for f in features])
        batch["labels"] = torch.tensor([f["labels"] for f in features])
        return batch


# ---------------------------------------------------------------------------
# 6. Multi-Sample Dropout Classification Head
# ---------------------------------------------------------------------------

class MultiSampleDropoutClassifier(nn.Module):
    def __init__(
        self,
        hidden_size: int,
        num_labels: int,
        dropout_rates: Tuple[float, ...] = (0.1, 0.2, 0.3, 0.4, 0.5),
    ):
        super().__init__()
        self.dense = nn.Linear(hidden_size, hidden_size)
        self.dropouts = nn.ModuleList([nn.Dropout(p) for p in dropout_rates])
        self.out_proj = nn.Linear(hidden_size, num_labels)

    def forward(self, features: torch.Tensor, **kwargs) -> torch.Tensor:
        x = features[:, 0, :] if features.dim() == 3 else features
        x = self.dropouts[0](x)
        x = self.dense(x)
        x = torch.tanh(x)
        logits = torch.stack([self.out_proj(drop(x)) for drop in self.dropouts], dim=0).mean(dim=0)
        return logits


# ---------------------------------------------------------------------------
# 7. Code-Aware Data Augmentation (Semantic-preserving)
# ---------------------------------------------------------------------------

class CodeAugmenter:
    _HUMAN_VARS = list("abcdefghijklmnopqrstuvwxyz")

    def __init__(self, aug_prob: float = 0.3, seed: int = 42):
        self.aug_prob = aug_prob
        self.rng = random.Random(seed)

    def strip_comments(self, code: str) -> str:
        lines = code.split("\n")
        cleaned = []
        for line in lines:
            stripped = re.sub(r"(?<!['\"])#.*$", "", line).rstrip()
            cleaned.append(stripped)
        return "\n".join(cleaned)

    def normalize_identifiers(self, code: str) -> str:
        code = re.sub(r"\b([a-z][a-z0-9]*_){3,}[a-z][a-z0-9]*\b", "var_name", code)
        return code

    def add_blank_lines(self, code: str) -> str:
        lines = code.split("\n")
        new_lines = []
        for line in lines:
            new_lines.append(line)
            if line.strip() and self.rng.random() < 0.15:
                new_lines.append("")
        return "\n".join(new_lines)

    def remove_blank_lines(self, code: str) -> str:
        return re.sub(r"\n{3,}", "\n\n", code)

    def augment(self, code: str) -> str:
        if self.rng.random() > self.aug_prob:
            return code
        transforms = [self.strip_comments, self.normalize_identifiers, self.add_blank_lines, self.remove_blank_lines]
        chosen = self.rng.sample(transforms, k=self.rng.randint(1, 2))
        for fn in chosen:
            code = fn(code)
        return code


# ---------------------------------------------------------------------------
# 8. Layer-Wise Learning Rate Decay (LLRD)
# ---------------------------------------------------------------------------

def get_llrd_optimizer(
    model: nn.Module,
    base_lr: float,
    weight_decay: float = 0.01,
    layer_decay: float = 0.9,
    num_layers: int = 12,
) -> AdamW:
    no_decay = {"bias", "LayerNorm.weight", "layer_norm.weight"}
    optimizer_grouped_parameters = []

    head_params = [
        p for n, p in model.named_parameters()
        if any(h in n for h in ("classifier", "pooler", "multi_sample"))
        and p.requires_grad
    ]
    optimizer_grouped_parameters.append({"params": head_params, "lr": base_lr, "weight_decay": 0.0})

    for layer_idx in range(num_layers - 1, -1, -1):
        layer_lr = base_lr * (layer_decay ** (num_layers - 1 - layer_idx))
        layer_params_decay = [
            p for n, p in model.named_parameters()
            if f"layer.{layer_idx}." in n
            and not any(nd in n for nd in no_decay)
            and p.requires_grad
        ]
        layer_params_no_decay = [
            p for n, p in model.named_parameters()
            if f"layer.{layer_idx}." in n
            and any(nd in n for nd in no_decay)
            and p.requires_grad
        ]
        optimizer_grouped_parameters.append(
            {"params": layer_params_decay, "lr": layer_lr, "weight_decay": weight_decay}
        )
        optimizer_grouped_parameters.append(
            {"params": layer_params_no_decay, "lr": layer_lr, "weight_decay": 0.0}
        )

    embed_lr = base_lr * (layer_decay ** num_layers)
    embed_params_decay = [
        p for n, p in model.named_parameters()
        if "embeddings" in n and not any(nd in n for nd in no_decay) and p.requires_grad
    ]
    embed_params_no_decay = [
        p for n, p in model.named_parameters()
        if "embeddings" in n and any(nd in n for nd in no_decay) and p.requires_grad
    ]
    optimizer_grouped_parameters.append(
        {"params": embed_params_decay, "lr": embed_lr, "weight_decay": weight_decay}
    )
    optimizer_grouped_parameters.append(
        {"params": embed_params_no_decay, "lr": embed_lr, "weight_decay": 0.0}
    )

    return AdamW(optimizer_grouped_parameters, lr=base_lr, eps=1e-8)


# ---------------------------------------------------------------------------
# 9. Custom Trainer with Advanced Features
# ---------------------------------------------------------------------------

class CodeDetectionCustomTrainer(Trainer):
    def __init__(
        self,
        loss_fn: Optional[nn.Module] = None,
        use_llrd: bool = True,
        layer_decay: float = 0.9,
        num_layers: int = 12,
        use_r_drop: bool = False,
        r_drop_weight: float = 0.5,
        use_triplet: bool = False,
        triplet_margin: float = 1.0,
        triplet_weight: float = 0.1,
        use_mixcode: bool = False,
        mixcode_alpha: float = 0.2,
        *args,
        **kwargs,
    ):
        super().__init__(*args, **kwargs)
        self.loss_fn = loss_fn
        self.use_llrd = use_llrd
        self.layer_decay = layer_decay
        self.num_layers = num_layers
        self.use_r_drop = use_r_drop
        self.r_drop_weight = r_drop_weight
        self.use_triplet = use_triplet
        self.triplet_margin = triplet_margin
        self.triplet_weight = triplet_weight
        self.use_mixcode = use_mixcode
        self.mixcode_alpha = mixcode_alpha

        if self.use_triplet:
            self._add_projection_head()

    def _add_projection_head(self):
        hidden_size = self.model.config.hidden_size
        self.projection_head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, 128)
        )
        self.projection_head.to(self.model.device)
        self.model.add_module("projection_head", self.projection_head)

    def compute_loss(
        self,
        model: nn.Module,
        inputs: Dict,
        return_outputs: bool = False,
        num_items_in_batch: Optional[int] = None,
    ):
        labels = inputs.pop("labels")
        if self.use_r_drop and self.model.training:
            inputs1 = {k: v.clone() if torch.is_tensor(v) else v for k, v in inputs.items()}
            inputs2 = {k: v.clone() if torch.is_tensor(v) else v for k, v in inputs.items()}
            outputs1 = model(**inputs1)
            outputs2 = model(**inputs2)
            logits1, logits2 = outputs1.logits, outputs2.logits
            if self.loss_fn is not None:
                ce_loss1 = self.loss_fn(logits1, labels)
                ce_loss2 = self.loss_fn(logits2, labels)
                ce_loss = (ce_loss1 + ce_loss2) / 2
            else:
                ce_loss1 = F.cross_entropy(logits1, labels)
                ce_loss2 = F.cross_entropy(logits2, labels)
                ce_loss = (ce_loss1 + ce_loss2) / 2
            kl_loss = r_drop_loss(logits1, logits2)
            total_loss = ce_loss + self.r_drop_weight * kl_loss
        else:
            outputs = model(**inputs)
            logits = outputs.logits
            if self.loss_fn is not None:
                total_loss = self.loss_fn(logits, labels)
            else:
                total_loss = F.cross_entropy(logits, labels)

        if self.use_triplet and self.model.training and hasattr(self, "projection_head"):
            with torch.set_grad_enabled(True):
                outputs_hidden = model(**inputs, output_hidden_states=True)
                last_hidden = outputs_hidden.hidden_states[-1]
                cls_emb = last_hidden[:, 0, :]
                proj_emb = self.projection_head(cls_emb)
                triplet_loss_fn = BatchHardTripletLoss(margin=self.triplet_margin)
                triplet_loss = triplet_loss_fn(proj_emb, labels)
                total_loss = total_loss + self.triplet_weight * triplet_loss

        return (total_loss, outputs) if return_outputs else total_loss

    def create_optimizer(self):
        if self.use_llrd:
            self.optimizer = get_llrd_optimizer(
                model=self.model,
                base_lr=self.args.learning_rate,
                weight_decay=self.args.weight_decay,
                layer_decay=self.layer_decay,
                num_layers=self.num_layers,
            )
            return self.optimizer
        return super().create_optimizer()

    def training_step(self, model: nn.Module, inputs: Dict[str, Union[torch.Tensor, Any]], num_items_in_batch: Optional[int] = None) -> torch.Tensor:
        loss = super().training_step(model, inputs, num_items_in_batch)
        if self.args.max_grad_norm is not None and self.args.max_grad_norm > 0:
            zclip_grad_norm_(model.parameters(), self.args.max_grad_norm)
        return loss


# ---------------------------------------------------------------------------
# 10. Main Trainer Class
# ---------------------------------------------------------------------------

class CodeDetectionTrainer:
    def __init__(
        self,
        max_length: int = 512,
        model_name: str = "microsoft/unixcoder-base",
        seed: int = 42,
        use_focal_loss: bool = True,
        use_label_smoothing: bool = True,
        smoothing: float = 0.05,
        use_multi_sample_dropout: bool = True,
        use_llrd: bool = True,
        layer_decay: float = 0.9,
        use_augmentation: bool = True,
        aug_prob: float = 0.25,
        fp16: bool = False,
        bf16: bool = False,
        use_r_drop: bool = False,
        r_drop_weight: float = 0.5,
        use_triplet: bool = False,
        triplet_margin: float = 1.0,
        triplet_weight: float = 0.1,
        use_mixcode: bool = False,
        mixcode_alpha: float = 0.2,
        use_peft: bool = False,
        peft_r: int = 8,
        peft_alpha: int = 32,
    ):
        self.max_length = max_length
        self.model_name = model_name
        self.seed = seed
        self.use_focal_loss = use_focal_loss
        self.use_label_smoothing = use_label_smoothing
        self.smoothing = smoothing
        self.use_multi_sample_dropout = use_multi_sample_dropout
        self.use_llrd = use_llrd
        self.layer_decay = layer_decay
        self.use_augmentation = use_augmentation
        self.aug_prob = aug_prob
        self.fp16 = fp16
        self.bf16 = bf16
        self.use_r_drop = use_r_drop
        self.r_drop_weight = r_drop_weight
        self.use_triplet = use_triplet
        self.triplet_margin = triplet_margin
        self.triplet_weight = triplet_weight
        self.use_mixcode = use_mixcode
        self.mixcode_alpha = mixcode_alpha
        self.use_peft = use_peft
        self.peft_r = peft_r
        self.peft_alpha = peft_alpha

        self.tokenizer = None
        self.model = None
        self.trainer = None
        self.num_labels = None
        self._augmenter = CodeAugmenter(aug_prob=aug_prob, seed=seed)

    def load_and_prepare_data(self) -> Tuple[Dataset, Dataset]:
        logger.info("Loading datasets from Hugging Face Hub...")
        try:
            train_dataset = load_dataset("DaniilOr/SemEval-2026-Task13", "A", split="train")
            val_dataset = load_dataset("DaniilOr/SemEval-2026-Task13", "A", split="validation")

            if "code" not in train_dataset.column_names or "label" not in train_dataset.column_names:
                raise ValueError("Dataset must contain 'code' and 'label' columns")

            self.num_labels = len(set(train_dataset["label"]))
            logger.info(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Labels: {self.num_labels}")
            return train_dataset, val_dataset
        except Exception as e:
            logger.error(f"Error loading dataset: {e}")
            raise

    def preprocess_code(self, code_str: str, augment: bool = False) -> str:
        code_str = code_str.lstrip("\ufeff\u200b\u200c\u200d")
        code_str = re.sub(r"\r\n|\r", "\n", code_str)
        code_str = "\n".join(line.rstrip() for line in code_str.split("\n"))
        code_str = re.sub(r"\n{3,}", "\n\n", code_str)
        code_str = re.sub(r"[ \t]+", " ", code_str)

        if augment and self.use_augmentation:
            code_str = self._augmenter.augment(code_str)

        return code_str.strip()

    def initialize_model_and_tokenizer(self) -> None:
        logger.info(f"Initialising backbone: {self.model_name}")
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(
            self.model_name,
            num_labels=self.num_labels,
            problem_type="single_label_classification",
            ignore_mismatched_sizes=True,
        )

        if self.use_peft and PEFT_AVAILABLE:
            logger.info("Applying LoRA (PEFT) to the model.")
            lora_config = LoraConfig(
                task_type=TaskType.SEQ_CLS,
                r=self.peft_r,
                lora_alpha=self.peft_alpha,
                target_modules=["query", "value"],
                lora_dropout=0.1,
                bias="none",
            )
            self.model = get_peft_model(self.model, lora_config)
            self.model.print_trainable_parameters()

        if self.use_multi_sample_dropout and not self.use_peft:
            hidden_size = self.model.config.hidden_size
            logger.info(f"Replacing classifier with MultiSampleDropoutClassifier")
            self.model.classifier = MultiSampleDropoutClassifier(
                hidden_size=hidden_size,
                num_labels=self.num_labels,
            )

        total_params = sum(p.numel() for p in self.model.parameters() if p.requires_grad)
        logger.info(f"Trainable parameters: {total_params:,}")

    def tokenize_function(self, examples: Dict, is_train: bool = False) -> Dict:
        cleaned = [self.preprocess_code(c, augment=is_train) for c in examples["code"]]
        return self.tokenizer(cleaned, truncation=True, max_length=self.max_length, padding=False)

    def prepare_datasets(self, train_dataset: Dataset, val_dataset: Dataset) -> Tuple[Dataset, Dataset]:
        logger.info("Preparing datasets...")
        columns_to_remove = [col for col in ["code", "generator", "language"] if col in train_dataset.column_names]

        train_dataset = train_dataset.map(
            lambda ex: self.tokenize_function(ex, is_train=True),
            batched=True,
            batch_size=2000,
            num_proc=2,
            remove_columns=columns_to_remove,
            desc="Tokenising train",
        )
        val_dataset = val_dataset.map(
            lambda ex: self.tokenize_function(ex, is_train=False),
            batched=True,
            batch_size=2000,
            num_proc=2,
            remove_columns=columns_to_remove,
            desc="Tokenising val",
        )
        train_dataset = train_dataset.rename_column("label", "labels")
        val_dataset = val_dataset.rename_column("label", "labels")
        return train_dataset, val_dataset

    def compute_metrics(self, eval_pred) -> Dict:
        predictions, labels = eval_pred
        predictions = torch.argmax(torch.tensor(predictions), dim=1).numpy()
        macro_f1 = precision_recall_fscore_support(labels, predictions, average="macro")[2]
        accuracy = accuracy_score(labels, predictions)
        precision, recall, f1_weighted, _ = precision_recall_fscore_support(labels, predictions, average="weighted")
        return {
            "macro_f1": macro_f1,
            "accuracy": accuracy,
            "f1_weighted": f1_weighted,
            "precision": precision,
            "recall": recall,
        }

    def _build_loss_fn(self) -> nn.Module:
        if self.use_focal_loss and self.use_label_smoothing:
            logger.info("Loss: FocalLossWithSmoothing")
            return FocalLossWithSmoothing(alpha=1.0, gamma=2.0, smoothing=self.smoothing)
        elif self.use_focal_loss:
            logger.info("Loss: FocalLoss")
            return FocalLoss(alpha=1, gamma=2)
        elif self.use_label_smoothing:
            logger.info("Loss: LabelSmoothingCrossEntropy")
            return LabelSmoothingCrossEntropy(smoothing=self.smoothing)
        else:
            logger.info("Loss: standard cross-entropy")
            return None

    def train(
        self,
        output_dir: str = "./results",
        num_epochs: int = 3,
        batch_size: int = 16,
        learning_rate: float = 2e-5,
        resume_from_checkpoint: Optional[str] = None,
    ) -> CodeDetectionCustomTrainer:
        logger.info("Loading data...")
        train_dataset, val_dataset = self.load_and_prepare_data()

        logger.info("Initialising model...")
        self.initialize_model_and_tokenizer()

        logger.info("Preparing datasets...")
        train_dataset, val_dataset = self.prepare_datasets(train_dataset, val_dataset)

        steps_per_epoch = max(1, len(train_dataset) // batch_size)
        total_steps = num_epochs * steps_per_epoch
        warmup_steps = max(1, int(total_steps * 0.06))
        num_layers = getattr(self.model.config, "num_hidden_layers", 12)

        training_args = TrainingArguments(
            output_dir=output_dir,
            num_train_epochs=num_epochs,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size * 2,
            weight_decay=0.01,
            logging_steps=10,
            eval_strategy="steps",
            eval_steps=200,
            save_strategy="steps",
            save_steps=200,
            load_best_model_at_end=True,
            metric_for_best_model="macro_f1",
            greater_is_better=True,
            remove_unused_columns=False,
            learning_rate=learning_rate,
            lr_scheduler_type="cosine_with_restarts",
            lr_scheduler_kwargs={"num_cycles": 2},
            warmup_steps=warmup_steps,
            fp16=self.fp16,
            bf16=self.bf16,
            gradient_checkpointing=True,
            dataloader_num_workers=2,
            max_grad_norm=1.0,
            save_total_limit=2,
            report_to=["wandb"],
            seed=self.seed,
        )

        data_collator = DataCollatorWithPadding(tokenizer=self.tokenizer)
        loss_fn = self._build_loss_fn()

        self.trainer = CodeDetectionCustomTrainer(
            loss_fn=loss_fn,
            use_llrd=self.use_llrd,
            layer_decay=self.layer_decay,
            num_layers=num_layers,
            use_r_drop=self.use_r_drop,
            r_drop_weight=self.r_drop_weight,
            use_triplet=self.use_triplet,
            triplet_margin=self.triplet_margin,
            triplet_weight=self.triplet_weight,
            use_mixcode=self.use_mixcode,
            mixcode_alpha=self.mixcode_alpha,
            model=self.model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            data_collator=data_collator,
            compute_metrics=self.compute_metrics,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
        )

        logger.info("***** Training Configuration *****")
        logger.info(f"  Backbone            : {self.model_name}")
        logger.info(f"  Max epochs          : {num_epochs}")
        logger.info(f"  Batch size          : {batch_size}")
        logger.info(f"  Learning rate       : {learning_rate}")
        logger.info(f"  LR scheduler        : cosine_with_restarts (num_cycles=2)")
        logger.info(f"  Warmup steps        : {warmup_steps} / {total_steps}")
        logger.info(f"  LLRD                : {self.use_llrd} (decay={self.layer_decay})")
        logger.info(f"  Mixed precision     : fp16={self.fp16}, bf16={self.bf16}")
        logger.info(f"  Augmentation        : {self.use_augmentation} (p={self.aug_prob})")
        logger.info(f"  Multi-sample dropout: {self.use_multi_sample_dropout}")
        logger.info(f"  R-Drop              : {self.use_r_drop} (weight={self.r_drop_weight})")
        logger.info(f"  Triplet loss        : {self.use_triplet} (margin={self.triplet_margin}, weight={self.triplet_weight})")
        logger.info(f"  MixCode             : {self.use_mixcode} (alpha={self.mixcode_alpha})")
        logger.info(f"  PEFT (LoRA)         : {self.use_peft}")
        logger.info(f"  Gradient checkpoint : True")
        if resume_from_checkpoint:
            logger.info(f"  Resuming from checkpoint: {resume_from_checkpoint}")

        self.trainer.train(resume_from_checkpoint=resume_from_checkpoint)
        logger.info(f"Training complete. Output: {output_dir}")
        return self.trainer

    def run_full_pipeline(
        self,
        output_dir: str = "./results",
        num_epochs: int = 3,
        batch_size: int = 16,
        learning_rate: float = 2e-5,
        resume_from_checkpoint: Optional[str] = None,
    ) -> CodeDetectionCustomTrainer:
        try:
            self.train(
                output_dir=output_dir,
                num_epochs=num_epochs,
                batch_size=batch_size,
                learning_rate=learning_rate,
                resume_from_checkpoint=resume_from_checkpoint,
            )
            logger.info(f"Best model saved to: {os.path.join(output_dir, 'best_model')}")
            return self.trainer
        except Exception as e:
            logger.error(f"Pipeline error: {e}")
            raise

    def evaluate(self, eval_dataset=None):
        if self.trainer is None:
            logger.error("No trainer found. Run train() first.")
            return None
        logger.info("Evaluating model...")
        predictions = self.trainer.predict(eval_dataset)
        y_pred = torch.argmax(torch.tensor(predictions.predictions), dim=1).numpy()
        y_true = predictions.label_ids
        logger.info("\n***** Classification Report *****")
        print(classification_report(y_true, y_pred, digits=4))
        return predictions

In [8]:
import random
from datasets import concatenate_datasets

OUTPUT_DIR = "taskA-codebert-base-focal"
MODEL_NAME = "microsoft/codebert-base"

# --- COUNTERING IMBALANCED DATASET (OVERSAMPLING) ---
# We monkey-patch the `prepare_datasets` method to duplicate minority class
# samples in the training set so both classes have equal representation.

# Guard: Only patch once to prevent recursion on re-execution
if not hasattr(CodeDetectionTrainer.prepare_datasets, '_is_patched'):
    original_prepare_datasets = CodeDetectionTrainer.prepare_datasets

    def prepare_datasets_balanced(self, train_dataset, val_dataset):
        # Run original tokenization/formatting
        train_ds, val_ds = original_prepare_datasets(self, train_dataset, val_dataset)

        # Balance training dataset ONLY
        labels = train_ds["labels"]
        class_0_idx = [i for i, l in enumerate(labels) if l == 0]
        class_1_idx = [i for i, l in enumerate(labels) if l == 1]

        logger.info(f"Before balancing - Class 0: {len(class_0_idx)}, Class 1: {len(class_1_idx)}")

        if len(class_1_idx) < len(class_0_idx):
            # Oversample class 1 natively
            diff = len(class_0_idx) - len(class_1_idx)
            oversample_idx = random.choices(class_1_idx, k=diff)
            oversample_ds = train_ds.select(oversample_idx)
            train_ds = concatenate_datasets([train_ds, oversample_ds]).shuffle(seed=self.seed)
            logger.info(f"After balancing - Added {diff} samples to Class 1. Total: {len(train_ds)}")
        elif len(class_0_idx) < len(class_1_idx):
            # Oversample class 0
            diff = len(class_1_idx) - len(class_0_idx)
            oversample_idx = random.choices(class_0_idx, k=diff)
            oversample_ds = train_ds.select(oversample_idx)
            train_ds = concatenate_datasets([train_ds, oversample_ds]).shuffle(seed=self.seed)
            logger.info(f"After balancing - Added {diff} samples to Class 0. Total: {len(train_ds)}")

        return train_ds, val_ds

    # Mark the patched method so we don't patch again
    prepare_datasets_balanced._is_patched = True
    CodeDetectionTrainer.prepare_datasets = prepare_datasets_balanced
    logger.info("✅ Dataset balancing monkey-patch applied")
else:
    logger.info("⚠️ Dataset balancing already patched, skipping re-patch")

# Initialize trainer
trainer_obj = CodeDetectionTrainer(
    max_length=512,
    model_name=MODEL_NAME,
    seed=42,
    use_focal_loss=True,
    use_label_smoothing=True,
    use_multi_sample_dropout=True,
    use_llrd=True,
    use_augmentation=True,
    use_r_drop=True,
    use_triplet=True,
    use_mixcode=True,
    use_peft=False,
    fp16=torch.cuda.is_available()
)

2026-04-13 16:44:35,267 - INFO - ✅ Dataset balancing monkey-patch applied
INFO:__main__:✅ Dataset balancing monkey-patch applied


In [ ]:
# Run training pipeline
import os
import wandb
wandb.init()
os.environ["WANDB_DISABLED"] = "false"


trainer_obj.run_full_pipeline(
    output_dir=OUTPUT_DIR,
    num_epochs=1,
    batch_size=64,
    learning_rate=2e-5,
    resume_from_checkpoint=None,
)

2026-04-13 16:44:42,753 - INFO - Loading data...
INFO:__main__:Loading data...
2026-04-13 16:44:42,756 - INFO - Loading datasets from Hugging Face Hub...
INFO:__main__:Loading datasets from Hugging Face Hub...


README.md:   0%|          | 0.00/801 [00:00<?, ?B/s]

task_a/task_a_training_set_1.parquet:   0%|          | 0.00/203M [00:00<?, ?B/s]

task_a/task_a_validation_set.parquet:   0%|          | 0.00/40.5M [00:00<?, ?B/s]

task_a/task_a_test_set_sample.parquet:   0%|          | 0.00/593k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at microsoft/codebert-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
2026-04-13 16:45:05,437 - INFO - Replacing classifier with MultiSampleDropoutClassifier
INFO:__main__:Replacing classifier with MultiSampleDropoutClassifier
2026-04-13 16:45:05,444 - INFO - Trainable parameters: 124,647,170
INFO:__main__:Trainable parameters: 124,647,170
2026-04-13 16:45:05,446 - INFO - Preparing datasets...
INFO:__main__:Preparing datasets...
2026-04-13 16:45:05,448 - INFO - Preparing datasets...
INFO:__main__:Preparing datasets...


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Tokenising train (num_proc=2):   0%|          | 0/500000 [00:00<?, ? examples/s]

In [ ]:
!ls -lart {OUTPUT_DIR}

In [ ]:
from huggingface_hub import HfApi

try:
    # Define repository name
    repo_name = f"dzungpham/SLA-SemEval-challenge"

    api = HfApi()
    logger.info(f"Creating/checking repo: {repo_name}")
    api.create_repo(repo_id=repo_name, repo_type="model", exist_ok=True)

    logger.info(f"Uploading {OUTPUT_DIR} to Hugging Face Hub...")
    # This uploads everything in the OUTPUT_DIR (checkpoints, best model, etc.)
    api.upload_folder(
        folder_path=OUTPUT_DIR,
        repo_id=repo_name,
        repo_type="model",
        commit_message="Upload training checkpoints and best model"
    )
    logger.info(f"Upload complete! Model is available at: https://huggingface.co/{repo_name}")
except Exception as e:
    logger.error(f"Failed to upload to Hugging Face: {e}")
    logger.info("Ensure you are authenticated with write access and have set your HF_TOKEN correctly.")

In [ ]:
from itertools import chain
from tqdm.auto import tqdm
import re
import torch
from datasets import load_dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from sklearn.metrics import classification_report, precision_recall_fscore_support

def preprocess_code(code_str):
    """Preprocess code string."""
    code_str = re.sub(r'[\r\n]+', '\n', code_str)
    code_str = re.sub(r'[ \t]+', ' ', code_str)
    return code_str

@torch.no_grad()
def predict_on_test(checkpoint_dir, output_path, max_length=512, batch_size=16):
    """
    Load model from checkpoint and perform predictions on test set.

    Args:
        checkpoint_dir: Path to the checkpoint directory containing model files
        output_path: Path to save predictions CSV
        max_length: Maximum sequence length for tokenization
        batch_size: Batch size for inference
    """
    device = "cuda" if torch.cuda.is_available() else "cpu"

    logger.info(f"Loading model and tokenizer from: {checkpoint_dir}")

    # Load model and tokenizer from checkpoint
    try:
        model = AutoModelForSequenceClassification.from_pretrained(checkpoint_dir)
        tokenizer = AutoTokenizer.from_pretrained(checkpoint_dir)
    except Exception as e:
        logger.error(f"Error loading checkpoint from {checkpoint_dir}: {e}")
        raise

    model = model.to(device)
    model.eval()
    logger.info(f"Model loaded on device: {device}")

    # Load test dataset from Hugging Face Hub
    logger.info("Loading test dataset...")
    ds = load_dataset("DaniilOr/SemEval-2026-Task13", "A", split="test", streaming=True)
    it = iter(ds)
    first = next(it)
    stream = chain([first], it)

    def batcher(iterator, bs):
        """Helper function to create batches from iterator."""
        buf = []
        for ex in iterator:
            buf.append(ex)
            if len(buf) == bs:
                yield buf
                buf = []
        if buf:
            yield buf

    all_preds = []
    all_targets = []

    # Perform inference on test set
    with open(output_path, "w") as f:
        f.write("id,label\n")
        logger.info("Predicting on test set...")

        for batch in tqdm(batcher(stream, batch_size), desc="Predicting"):
            # Preprocess codes
            codes = [preprocess_code(row["code"]) for row in batch]

            # Tokenize
            enc = tokenizer(
                codes,
                truncation=True,
                padding=True,
                max_length=max_length,
                return_tensors="pt",
            )
            input_ids = enc["input_ids"].to(device)
            attention_mask = enc["attention_mask"].to(device)

            # Get predictions
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            pred_labels = logits.argmax(dim=-1).cpu().tolist()

            all_preds.extend(pred_labels)

            if "label" in batch[0]:
                all_targets.extend([row["label"] for row in batch])

            # Write to CSV
            for row, pred in zip(batch, pred_labels):
                row_id = row.get("id", "unknown")
                f.write(f"{row_id},{pred}\n")

    logger.info(f"Predictions saved to {output_path}")

    # Calculate Evaluation Metrics
    if all_targets:
        logger.info("\n***** Test Set Evaluation Metrics *****")
        print(classification_report(all_targets, all_preds, digits=4))

        macro_f1 = precision_recall_fscore_support(all_targets, all_preds, average="macro")[2]
        logger.info(f"Macro F1-score: {macro_f1:.4f}")

# ============== Prediction Configuration ==============
# Choose which checkpoint to use for prediction:
# Option 1: Use the best model checkpoint (highest macro F1)
CHECKPOINT_PATH = "taskA-unixcoder-focal/checkpoint-2300"

# Option 2: Use the latest checkpoint
# CHECKPOINT_PATH = "taskA-unixcoder-focal/checkpoint-<step>"

OUT_CSV = "submission.csv"

# Run prediction using checkpoint
predict_on_test(
    checkpoint_dir=CHECKPOINT_PATH,
    output_path=OUT_CSV,
    max_length=512,
    batch_size=32,
)

logger.info(f"Predictions written to: {OUT_CSV}")

In [ ]:
import os
import zipfile
from pathlib import Path


def find_result_folders(base_path: Path, pattern_name: str) -> list[Path]:
    return [p for p in base_path.glob(pattern_name) if p.is_dir()]


def zip_folder(folder_path: Path, output_base_path: Path) -> bool:
    folder_name = folder_path.name
    zip_path = output_base_path / f"{folder_name}.zip"
    try:
        print(f"   -> Zipping folder: {folder_name}...")
        with zipfile.ZipFile(
            zip_path, mode="w", compression=zipfile.ZIP_DEFLATED
        ) as zipf:
            for file_path in folder_path.rglob("*"):
                if file_path.is_file():
                    arcname = file_path.relative_to(folder_path.parent)
                    zipf.write(file_path, arcname)
        print(f"  Created ZIP: {zip_path.name}")
        return True
    except Exception as exc:
        print(f"   Failed to zip {folder_name}: {exc}")
        return False


def zip_stats_results_folders(output_base_path: str, pattern_name: str) -> None:
    base = Path(output_base_path)
    base.mkdir(parents=True, exist_ok=True)
    result_folders = find_result_folders(base, pattern_name)
    if not result_folders:
        print(f"⚠️ No folders matching {pattern_name} found in '{output_base_path}'.")
        return
    print(f"Found {len(result_folders)} result folder(s) to zip.")
    successful = sum(1 for folder in result_folders if zip_folder(folder, base))
    print(
        f"\nDONE! Successfully zipped {successful} out of {len(result_folders)} folder(s)."
    )


if __name__ == "__main__":
    try:
        print("📦 Output Path:", OUTPUT_PATH)
        pattern_names = ["*checkpoint*"]
        output_root = os.getenv("OUTPUT_PATH") or globals().get("OUTPUT_PATH")
        if not output_root:
            raise ValueError("OUTPUT_PATH not defined")
        for pattern_name in pattern_names:
            zip_stats_results_folders(
                output_base_path=OUTPUT_PATH, pattern_name=pattern_name
            )
    except Exception as e:
        print(f"❌ An error occurred: {e}")

In [ ]:
if is_colab:
    from google.colab import files
    files.download("/content/submission.csv")
from IPython.display import FileLink
FileLink("submission.csv")